# FantasyData.com Weekly Stats Ingestion

Pull weekly NFL stats from FantasyData.com API and land in bronze/silver Delta tables.

**Note:** Requires API key from FantasyData.com

In [0]:
import requests
import json
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

# TODO: Set your FantasyData.com API key
API_KEY = "YOUR_API_KEY_HERE"  # Get from https://fantasydata.com
BASE_URL = "https://api.fantasydata.net/v3/nfl"  # Adjust based on actual API

# TODO: Configure week and season (or fetch dynamically)
WEEK = 18
SEASON = 2025

print(f"📅 Fetching stats for Week {WEEK}, Season {SEASON} from FantasyData.com")

In [0]:
# TODO: Update endpoint and parameters based on FantasyData.com API docs
# Example structure - needs to be adjusted based on actual API

headers = {
    "Ocp-Apim-Subscription-Key": API_KEY
}

try:
    response = requests.get(
        f"{BASE_URL}/stats/json/PlayerGameStatsByWeek/{SEASON}/{WEEK}",
        headers=headers,
        timeout=30
    )
    response.raise_for_status()
    stats_data = response.json()
    
    print(f"Fetched {len(stats_data)} player records")
    
    # Convert to DataFrame rows
    rows = []
    for player_stat in stats_data:
        # TODO: Adjust field mapping based on actual API response
        player_id = str(player_stat.get('PlayerID', ''))
        fantasy_points = float(player_stat.get('FantasyPointsPPR', 0))
        
        rows.append(
            Row(
                player_id=player_id,
                week=WEEK,
                season=SEASON,
                fantasy_points=fantasy_points,
                stats=json.dumps(player_stat),
                source='fantasydata'
            )
        )
    
    stats_df = spark.createDataFrame(rows)
    display(stats_df)
    
except Exception as e:
    print(f"Error fetching data: {e}")
    print("Please update API_KEY and endpoint configuration")

In [0]:
# Write to bronze table using MERGE
bronze_df = stats_df.withColumn("ingested_at", F.current_timestamp())

# Create temp view for merge
bronze_df.createOrReplaceTempView("fantasydata_bronze_updates")

# Perform MERGE operation
spark.sql("""
  MERGE INTO main.fantasai.bronze_weekly_stats AS target
  USING fantasydata_bronze_updates AS source
  ON target.player_id = source.player_id 
    AND target.week = source.week 
    AND target.season = source.season
  WHEN MATCHED THEN
    UPDATE SET
      target.fantasy_points = source.fantasy_points,
      target.stats = source.stats,
      target.ingested_at = source.ingested_at
  WHEN NOT MATCHED THEN
    INSERT (player_id, week, season, fantasy_points, stats, ingested_at)
    VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.ingested_at)
""")

print(f"✓ Merged {bronze_df.count()} records from FantasyData.com into bronze_weekly_stats")

In [0]:
# Transform for silver
silver_df = (
    bronze_df
    .select(
        F.col("player_id").cast("string"),
        F.col("week").cast("int"),
        F.col("season").cast("int"),
        F.col("fantasy_points").cast("double"),
        F.col("stats").cast("string"),
        F.col("ingested_at"),
    )
    .dropDuplicates(["player_id", "week", "season"])
)

# Create temp view for merge
silver_df.createOrReplaceTempView("fantasydata_silver_updates")

# Perform MERGE operation
spark.sql("""
  MERGE INTO main.fantasai.silver_weekly_stats AS target
  USING fantasydata_silver_updates AS source
  ON target.player_id = source.player_id 
    AND target.week = source.week 
    AND target.season = source.season
  WHEN MATCHED THEN
    UPDATE SET
      target.fantasy_points = source.fantasy_points,
      target.stats = source.stats,
      target.ingested_at = source.ingested_at
  WHEN NOT MATCHED THEN
    INSERT (player_id, week, season, fantasy_points, stats, ingested_at)
    VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.ingested_at)
""")

print(f"✓ Merged {silver_df.count()} records from FantasyData.com into silver_weekly_stats")